In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrapping_urea_final(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Buscamos todas las filas de todas las tablas
        filas = soup.find_all('tr')
        datos_limpios = []

        for fila in filas:
            celdas = fila.find_all('td')
            # FILTRO CRÍTICO: Solo procesamos filas que tengan exactamente 3 columnas
            if len(celdas) == 3:
                fila_texto = [c.text.strip() for c in celdas]
                # Evitamos filas vacías o que no contengan números en la segunda columna
                if fila_texto[1] != "":
                    datos_limpios.append(fila_texto)

        # Si no encontramos nada con 3 columnas, devolvemos error
        if not datos_limpios:
            return "No se encontró información con el formato esperado (3 columnas)."

        # Creamos el DataFrame definiendo nosotros los nombres (así evitamos el error de los 700)
        columnas_nombres = ['Mes', 'Precio_CLP', 'Variacion']
        df = pd.DataFrame(datos_limpios, columns=columnas_nombres)

        # --- Limpieza de datos (Tipo de dato numérico) ---
        # Quitamos puntos de miles y cambiamos coma decimal
        df['Precio_CLP'] = (df['Precio_CLP']
                            .str.replace('.', '', regex=False)
                            .str.replace(',', '.', regex=False)
                            .astype(float))
        
        # Limpieza de la variación
        df['Variacion'] = (df['Variacion']
                           .str.replace('%', '', regex=False)
                           .str.replace(',', '.', regex=False)
                           .replace('-', '0')
                           .astype(float))

        return df

    except Exception as e:
        return f"Error en el proceso: {e}"

# --- Ejecución ---
url_urea = "https://www.indexmundi.com/es/precios-de-mercado/?mercancia=urea&meses=240&moneda=clp"
df_urea = scrapping_urea_final(url_urea)

if isinstance(df_urea, pd.DataFrame):
    print("✅ Datos extraídos correctamente:")
    print(df_urea.head(15))
    
    # Opcional: Guardar a CSV para tu entrega de Big Data
    # df_potasio.to_csv("precios_urea.csv", index=False)
else:
    print(f"❌ Error: {df_urea}")

✅ Datos extraídos correctamente:
          Mes  Precio_CLP  Variacion
0   mar. 2006    128560.1       0.00
1   abr. 2006    128038.3      -0.41
2   may. 2006    119470.6      -6.69
3   jun. 2006    113916.6      -4.65
4   jul. 2006    111097.5      -2.47
5   ago. 2006    113390.6       2.06
6   sep. 2006    115880.3       2.20
7   oct. 2006    111075.7      -4.15
8   nov. 2006    119883.0       7.93
9   dic. 2006    132658.2      10.66
10  ene. 2007    143360.5       8.07
11  feb. 2007    162343.8      13.24
12  mar. 2007    171309.1       5.52
13  abr. 2007    154899.6      -9.58
14  may. 2007    152999.8      -1.23


In [6]:
# Diccionario para mapear meses en español a números
meses_map = {
    'ene.': '01', 'feb.': '02', 'mar.': '03', 'abr.': '04', 
    'may.': '05', 'jun.': '06', 'jul.': '07', 'ago.': '08', 
    'sep.': '09', 'oct.': '10', 'nov.': '11', 'dic.': '12'
}

# 1. Separamos el mes del año
df_split = df_urea['Mes'].str.split(' ', expand=True)

# 2. Reemplazamos el nombre del mes por su número
df_split[0] = df_split[0].map(meses_map)

# 3. Creamos la columna de fecha real
df_urea['Fecha'] = pd.to_datetime(df_split[1] + '-' + df_split[0] + '-01')

# 4. Ordenamos por fecha (por si el scraping los trajo desordenados)
df_urea = df_urea.sort_values('Fecha')

print(df_urea[['Fecha', 'Precio_CLP']].head())

       Fecha  Precio_CLP
0 2006-03-01    128560.1
1 2006-04-01    128038.3
2 2006-05-01    119470.6
3 2006-06-01    113916.6
4 2006-07-01    111097.5


In [4]:
import pandas as pd

# Asumiendo que df_potasio es el DataFrame que ya obtuvimos con el scrapping exitoso
def formatear_dataframe_big_data(df):
    # 1. Diccionario de mapeo para meses en español (formato IndexMundi)
    meses_map = {
        'ene.': '01', 'feb.': '02', 'mar.': '03', 'abr.': '04', 
        'may.': '05', 'jun.': '06', 'jul.': '07', 'ago.': '08', 
        'sep.': '09', 'oct.': '10', 'nov.': '11', 'dic.': '12'
    }

    # Creamos una copia para no alterar el original por accidente
    df_clean = df.copy()

    # 2. Procesamiento de Fecha
    # Separamos "mar. 2006" en ["mar.", "2006"]
    temp_split = df_clean['Mes'].str.split(' ', expand=True)
    
    # Mapeamos el nombre del mes a su número
    mes_num = temp_split[0].map(meses_map)
    año = temp_split[1]

    # Creamos la nueva columna Fecha (formato YYYY-MM-DD)
    df_clean['Fecha'] = pd.to_datetime(año + '-' + mes_num + '-01')

    # 3. Reorganizamos las columnas para que sea más legible
    # Dejamos la Fecha primero, luego el Precio y al final la Variación
    df_final = df_clean[['Fecha', 'Precio_CLP', 'Variacion']].sort_values('Fecha')

    return df_final

# --- Ejecución ---
df_listo = formatear_dataframe_big_data(df_potasio)

print("✅ DataFrame estructurado para análisis de series de tiempo:")
print(df_listo.head(15))

# Verificamos los tipos de datos (Dtype)
# print(df_listo.info())

✅ DataFrame estructurado para análisis de series de tiempo:
        Fecha  Precio_CLP  Variacion
0  2006-03-01    88569.12       0.00
1  2006-04-01    86652.16      -2.16
2  2006-05-01    87195.31       0.63
3  2006-06-01    90862.05       4.21
4  2006-07-01    93257.03       2.64
5  2006-08-01   103694.50      11.19
6  2006-09-01   103690.60       0.00
7  2006-10-01   102208.80      -1.43
8  2006-11-01   101493.00      -0.70
9  2006-12-01   101469.00      -0.02
10 2007-01-01   104139.30       2.63
11 2007-02-01   104386.30       0.24
12 2007-03-01   103658.90      -0.70
13 2007-04-01   102467.90      -1.15
14 2007-05-01   100475.70      -1.94
